In [1]:
# !hf download --repo-type dataset \
# --include 'malaysian-chinese_processed_trim-*.zip' \
# --local-dir './' \
# --max-workers 20 \
# Scicom-intl/Malaysian-Chinese-Emilia

In [2]:
# !hf download --repo-type dataset \
# --include '*.zip' \
# --local-dir './' \
# --max-workers 20 \
# mesolitica/Malaysian-Emilia-v2

In [4]:
import os

os.environ['OMP_NUM_THREADS'] = '1'
os.environ['OPENBLAS_NUM_THREADS'] = '1'

import malaya_speech
from glob import glob
import librosa
import soundfile as sf
import numpy as np
import os
from multiprocess import Pool
import itertools
from tqdm import tqdm

def chunks(l, n):
    for i in range(0, len(l), n):
        yield (l[i: i + n], i // n)

def multiprocessing(strings, function, cores=6, returned=True):
    df_split = chunks(strings, len(strings) // cores)
    pool = Pool(cores)
    pooled = pool.map(function, df_split)
    pool.close()
    pool.join()

    if returned:
        return list(itertools.chain(*pooled))
    
def new_path(f):
    splitted = f.split('/')
    base_folder = splitted[0] + '_trim'
    splitted = '/'.join([base_folder] + splitted[1:])
    return splitted

2025-12-05 10:34:47.865061: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1764930887.874350 2859180 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1764930887.878784 2859180 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1764930887.884124 2859180 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1764930887.884139 2859180 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1764930887.884140 2859180 computation_placer.cc:177] computation placer alr

In [8]:
from glob import glob

folders = glob('*_processed')
files = []
for f in folders:
    files.extend(glob(f'{f}/**/*.mp3', recursive = True))
files = [f for f in files if 'trim/' not in f]

In [9]:
len(files)

1572014

In [10]:
def loop(files):

    os.environ['OMP_NUM_THREADS'] = '1'
    os.environ['OPENBLAS_NUM_THREADS'] = '1'

    files, _ = files
    
    for f in tqdm(files):
        
        f_new = new_path(f)
        if os.path.exists(f_new):
            continue
        
        try:
            vad = malaya_speech.vad.webrtc(minimum_amplitude = 0)
            y, sr = sf.read(f)

            min_length = 0.4
            start_silent_trail = int(0.3 * sr)
            middle_silent_trail = int(min_length * sr / 2)
            middle_silent_trail, start_silent_trail

            y_= malaya_speech.resample(y, sr, 16000)
            y_ = malaya_speech.astype.float_to_int(y_)
            frames = malaya_speech.generator.frames(y, 30, sr)
            frames_ = list(malaya_speech.generator.frames(y_, 30, 16000, append_ending_trail = False))
            frames_webrtc = [(frames[no], vad(frame)) for no, frame in enumerate(frames_)]
            grouped_deep = malaya_speech.group.group_frames(frames_webrtc)
            r = []
            for no, g in enumerate(grouped_deep):
                if g[1]:
                    g = g[0].array
                else:
                    if no == 0:
                        g = g[0].array[-start_silent_trail:]
                    elif no == (len(grouped_deep) - 1):
                        g = g[0].array[:start_silent_trail]
                    else:
                        if g[0].duration >= min_length:
                            g = [g[0].array[:middle_silent_trail], g[0].array[-middle_silent_trail:]]
                            g = np.concatenate(g)
                        else:
                            g = g[0].array

                r.append(g)
            y_after = np.concatenate(r)
            
            os.makedirs(os.path.split(f_new)[0], exist_ok = True)
            sf.write(f_new, y_after, sr)
            
        except Exception as e:
            print(e)

In [11]:
data = loop((files[:1000], 0))

100%|██████████| 1000/1000 [01:54<00:00,  8.74it/s]


In [14]:
multiprocessing(files, loop, cores = 20, returned = False)

100%|██████████| 78600/78600 [2:33:03<00:00,  8.56it/s]


In [15]:
folders = glob('*_processed')
files = []
for f in folders:
    files.extend(glob(f'{f}_trim/**/*.mp3', recursive = True))
len(files)

1572014

In [17]:
repository = 'malaysia-ai/Malaysian-Emilia'
folder = 'output-audio_trim'

In [18]:
import zipfile
import time
from huggingface_hub import HfFileSystem
from huggingface_hub import HfApi
from tqdm import tqdm
api = HfApi()

partition_size = 10e+9

In [19]:
def loop(files):
    files, index = files
    current_index = 0
    api = HfApi()
    fs = HfFileSystem()
    total = 0
    temp = []
    for i in tqdm(range(len(files))):
        s = os.stat(files[i]).st_size
        if s + total >= partition_size:
            part_name = f"{folder}-{index}-{current_index}.zip"
                
            with zipfile.ZipFile(part_name, 'w', zipfile.ZIP_DEFLATED) as zipf:
                for f in temp:
                    zipf.write(f, arcname=f)

            while True:
                try:
                    api.upload_file(
                        path_or_fileobj=part_name,
                        path_in_repo=part_name,
                        repo_id=repository,
                        repo_type="dataset",
                    )
                    break
                except:
                    time.sleep(60)

            os.remove(part_name)
            
            current_index += 1
            temp = [files[i]]
            total = s
        else:
            temp.append(files[i])
            total += s
        
    if len(temp):
        part_name = f"{folder}-{index}-{current_index}.zip"

        with zipfile.ZipFile(part_name, 'w', zipfile.ZIP_DEFLATED) as zipf:
            for f in temp:
                zipf.write(f, arcname=f)

        while True:
            try:
                api.upload_file(
                    path_or_fileobj=part_name,
                    path_in_repo=part_name,
                    repo_id=repository,
                    repo_type="dataset",
                )
                break
            except:
                time.sleep(60)

        os.remove(part_name)

In [21]:
# multiprocessing(files, loop, cores = 10, returned = False)

In [2]:
from glob import glob

folders = glob('*_processed')
files = []
for f in folders:
    files.extend(glob(f'{f}_trim/**/*.mp3', recursive = True))
len(files)

1572014

In [3]:
import json

with open('malaysian-emilia-audio.json', 'w') as fopen:
    json.dump(files, fopen)

In [4]:
from huggingface_hub import hf_hub_download
hf_hub_download(
    repo_id="malaysia-ai/malaysian-emilia-old", 
    filename="reduce-sample-malaysian.parquet", 
    repo_type='dataset', local_dir='./')

/home/ubuntu/.local/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


'reduce-sample-malaysian.parquet'

In [5]:
import pandas as pd

df = pd.read_parquet('reduce-sample-malaysian.parquet')

In [8]:
rows = df.to_dict(orient='records')

In [9]:
rows[0]

{'reference_audio': 'klasik_processed/Engkau Laksana Bulan 🌙 [0JK9DbTBrDA]/Engkau Laksana Bulan 🌙 [0JK9DbTBrDA]_1.mp3',
 'reference_text': 'Berjumpa dan bercinta.',
 'target_audio': 'klasik_processed/Engkau Laksana Bulan 🌙 [0JK9DbTBrDA]/Engkau Laksana Bulan 🌙 [0JK9DbTBrDA]_2.mp3',
 'target_text': 'berjumpa dan bercinta.'}

In [ ]:
from datasets import Dataset

dataset = Dataset.from_list(rows)
dataset[0]

In [ ]:
dataset.push_to_hub('malaysia-ai/Multilingual-TTS', 'coral-v2')